# Jour 3 — Passer de Keras à PyTorch sans réapprendre le Deep Learning

## Objectifs

- distinguer les concepts du Deep Learning de la syntaxe d’un framework
- retrouver tenseurs, couches, loss, gradients et optimiseur dans PyTorch
- comprendre une boucle d’entraînement explicite
- entraîner le même type de réseau Dense que précédemment

## Un nouveau langage pour les mêmes opérations

![Même réseau exprimé avec Keras et PyTorch](../ressources/illustrations/jour_03/03_keras_pytorch_meme_reseau.png)

Keras automatise une grande partie de la boucle avec `compile` puis `fit`. PyTorch demande généralement d’écrire cette boucle. Cette différence rend le code un peu plus long, mais laisse voir chaque étape.

Changer de framework ne change pas le raisonnement : les données deviennent des tenseurs, le réseau produit une prédiction, la loss mesure l’erreur, la rétropropagation calcule les gradients et l’optimiseur modifie les poids.

## Le dictionnaire Keras ↔ PyTorch

| Concept | TensorFlow/Keras | PyTorch |
|---|---|---|
| tenseur | `tf.Tensor` | `torch.Tensor` |
| couche Dense | `Dense` | `nn.Linear` |
| activation ReLU | `activation="relu"` ou `ReLU` | `nn.ReLU` |
| modèle en chaîne | `Sequential` | `nn.Sequential` |
| loss multiclasse | `SparseCategoricalCrossentropy` | `nn.CrossEntropyLoss` |
| optimiseur | `tf.keras.optimizers.Adam` | `torch.optim.Adam` |
| rétropropagation | réalisée par `fit` | `loss.backward()` |
| mise à jour | réalisée par `fit` | `optimizer.step()` |
| mode évaluation | géré par `evaluate`/`predict` | `model.eval()` et `torch.no_grad()` |

Le mot **tenseur** garde le même sens : un tableau numérique possédant une forme et un type. Les objets TensorFlow et PyTorch ajoutent notamment le calcul automatique des gradients.

## Préparer exactement le même problème

Nous reprenons les chiffres manuscrits de 8 × 8 pixels. Pour rester concentrés sur PyTorch, nous utilisons un réseau Dense : chaque image est aplatie en un vecteur de 64 valeurs, puis classée parmi dix chiffres.

Un `TensorDataset` associe chaque tenseur d’entrée à son label. Un `DataLoader` découpe ensuite le jeu en batchs et mélange les exemples d’entraînement. Il joue donc le rôle de distributeur de mini-lots.

`reshape(-1, 64)` range chaque image dans une ligne de 64 pixels. Le `-1` demande à NumPy de déduire le nombre de lignes : les 1 797 images gardent leurs valeurs.

`torch.from_numpy` rend le tableau utilisable par PyTorch. `TensorDataset` associe entrées et réponses ; `DataLoader` distribue des lots. `shuffle=True` mélange le train, et le `generator` fixe ce hasard.

`iter(train_loader)` commence le parcours des lots ; `next(...)` prend le premier ; `[0]` choisit ses entrées plutôt que ses labels. Ces commandes inspectent un lot, elles n'entraînent pas le réseau.

La question et la forme du Dense sont reprises du jour 2. Le protocole présenté ici utilise tout le train avec vingt époques fixées ; son score n'est donc pas une comparaison contrôlée avec l'arrêt anticipé Keras.


In [ ]:
# NumPy gère les tableaux et les calculs numériques.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# PyTorch gère les tenseurs et l’apprentissage.
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


# Fixe le hasard de PyTorch.
torch.manual_seed(42)
torch.set_num_threads(1)
plt.style.use("seaborn-v0_8-whitegrid")
print("PyTorch", torch.__version__)

# Charge les images et leurs labels ensemble.
digits = load_digits()
# Range les 64 pixels de chaque image sur une ligne.
X = (digits.images.astype("float32") / 16.0).reshape(-1, 64)
y = digits.target.astype("int64")


In [ ]:
# Sépare ensemble les entrées et les réponses pour garder leur association.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Convertit le tableau NumPy en tenseur utilisable par PyTorch.
X_train_tensor = torch.from_numpy(X_train)
# Convertit le tableau NumPy en tenseur utilisable par PyTorch.
y_train_tensor = torch.from_numpy(y_train)
# Convertit le tableau NumPy en tenseur utilisable par PyTorch.
X_test_tensor = torch.from_numpy(X_test)
# Convertit le tableau NumPy en tenseur utilisable par PyTorch.
y_test_tensor = torch.from_numpy(y_test)

# Conserve les entrées et leurs réponses dans le même ordre.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
generator = torch.Generator().manual_seed(42)
# Distribue les exemples par lots et mélange le train.
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, generator=generator
)

print("Entrées train :", tuple(X_train_tensor.shape))
print("Labels train  :", tuple(y_train_tensor.shape))


In [ ]:
# Lecture des résultats : aucune modification des poids dans cette cellule.
print("Un batch      :", tuple(next(iter(train_loader))[0].shape))


## Définir le réseau et les outils d’apprentissage

`nn.Linear(64, 32)` relie les 64 pixels aux 32 neurones cachés. `nn.ReLU()` introduit la non-linéarité. La dernière couche produit dix **logits**, c’est-à-dire dix scores bruts.

`CrossEntropyLoss` attend directement ces logits et les labels entiers de 0 à 9. Elle réalise elle-même la transformation nécessaire ; ajouter un `Softmax` dans le modèle serait ici une erreur fréquente.

`model.parameters()` fournit les poids et biais que l'optimiseur pourra corriger. `lr=0.001` est le learning rate de départ. La dernière couche produit des logits ; ils vont directement à `CrossEntropyLoss`, qui réalise le calcul multiclasse adapté.


In [ ]:
model = nn.Sequential(
    # Apprend 32 combinaisons des 64 pixels.
    nn.Linear(64, 32),
    nn.ReLU(),
    # Produit les dix logits, sans appliquer softmax ici.
    nn.Linear(32, 10),
)

loss_function = nn.CrossEntropyLoss()
# Fournit à Adam les paramètres à corriger.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)


## Lire la boucle d’entraînement ligne par ligne

Pour chaque batch :

1. `optimizer.zero_grad()` efface les gradients du batch précédent ;
2. `model(batch_X)` effectue la propagation avant ;
3. la loss compare les logits aux labels ;
4. `loss.backward()` calcule les gradients par rétropropagation ;
5. `optimizer.step()` met les poids à jour.

Une **époque** est terminée lorsque tous les batchs ont été parcourus une fois. Nous mémorisons la loss moyenne pour vérifier que l’apprentissage progresse.

`model.train()` active le comportement d'entraînement. Dans la boucle, `loss.item()` extrait la valeur numérique d'une loss. La multiplier par le nombre d'exemples du lot, puis diviser la somme par le nombre total d'exemples, donne la moyenne de l'époque, même si le dernier lot est plus petit.

`losses.append(...)` ajoute cette moyenne à la liste utilisée pour le graphique. Les poids changent dans la boucle ; le graphique affiche seulement les mesures recueillies.


In [ ]:
losses = []
for epoch in range(20):
    # Active le comportement d’entraînement.
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        # Efface les gradients du lot précédent.
        optimizer.zero_grad()
        # Calcule les scores avant comparaison à la vraie réponse.
        logits = model(batch_X)
        # Mesure l’écart aux labels du lot.
        loss = loss_function(logits, batch_y)
        # Calcule les gradients par rétropropagation.
        loss.backward()
        # Applique une correction aux poids et aux biais.
        optimizer.step()
        # Pondère la moyenne du lot par son nombre d’exemples.
        total_loss += loss.item() * len(batch_X)

    # Obtient la moyenne de l’époque, y compris un dernier lot plus petit.
    epoch_loss = total_loss / len(train_dataset)
    losses.append(epoch_loss)

plt.plot(range(1, len(losses) + 1), losses)


In [ ]:
plt.xlabel("époque")
plt.ylabel("cross-entropy")
plt.title("Loss du réseau PyTorch")
# Affiche le résultat des calculs précédents.
plt.show()


## Entraîner et prédire sont deux modes différents

`model.train()` active le comportement d’entraînement de certaines couches comme Dropout. `model.eval()` active leur comportement d’inférence. Même si notre petit réseau n’utilise pas Dropout, prendre cette habitude évite des erreurs dans des modèles plus grands.

`torch.no_grad()` indique qu’aucune rétropropagation ne sera faite. PyTorch évite alors de conserver les informations nécessaires aux gradients, ce qui réduit le coût de l’inférence.

`model.eval()` active le comportement de prédiction. `torch.no_grad()` désactive le calcul des gradients dans le bloc indenté ; il évite des calculs inutiles, mais ne remplace pas `eval()`.

`argmax(dim=1)` choisit la classe dans chaque ligne de logits. `dim` est le nom PyTorch pour l'axe. La comparaison aux labels donne des booléens ; `.float()` les transforme en 1 et 0, `.mean()` calcule leur proportion et `.item()` récupère le nombre à afficher.


In [ ]:
# Active le comportement de prédiction.
model.eval()
# Évite le calcul des gradients pendant cette évaluation.
with torch.no_grad():
    # Calcule les scores avant comparaison à la vraie réponse.
    test_logits = model(X_test_tensor)
    # Position du score le plus élevé de chaque ligne.
    test_predictions = test_logits.argmax(dim=1)

# Position du score le plus élevé de chaque ligne.
test_accuracy = (test_predictions == y_test_tensor).float().mean().item()
print("Accuracy test :", round(test_accuracy, 3))


## Ce qu’il faut réellement retenir

La version PyTorch n’introduit pas une nouvelle théorie. Elle rend visibles les étapes que Keras orchestre dans `fit`. Les deux frameworks savent construire des réseaux Dense, des CNN, des RNN et bien d’autres architectures.

Il n’est pas nécessaire de connaître toutes leurs commandes par cœur. Il faut savoir retrouver : les données, le modèle, la loss, l’optimiseur, la boucle d’entraînement et l’évaluation.

## À vous de jouer — obtenir des probabilités

Pour les cinq premières images du test :

1. placez le modèle en mode évaluation ;
2. calculez les logits sans gradient ;
3. utilisez `torch.softmax(..., dim=1)` pour obtenir des probabilités ;
4. affichez la classe prédite, la vraie classe et la probabilité maximale.

In [ ]:
# 1. Placez le modèle en mode évaluation.
# 2. Désactivez le calcul des gradients avec torch.no_grad().
# 3. Calculez les logits des cinq premières images de test.
# 4. Transformez-les en probabilités avec torch.softmax(..., dim=1).
# 5. Affichez classes prédites, vraies classes et probabilités maximales.
pass


## À retenir

Keras et PyTorch changent la manière d’écrire la boucle, pas les fondations du Deep Learning. Si vous savez expliquer ce qui se passe entre un batch, une loss et une mise à jour des poids, vous pouvez apprendre l’un ou l’autre framework sans repartir de zéro.